# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) dataset using the `mlcroissant` library. Following the Croissant schema, we will:

- Load and review metadata
- Explore record sets and fields by their `@id`
- Extract records for analysis
- Conduct exploratory data analysis (EDA)
- Visualize selected aspects

### Dataset Source
Source: [Croissant schema URL](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", getattr(metadata, 'identifier', None))
print("Authors (by @id):", getattr(metadata, 'author', None))

## 2. Data Overview
Review available record sets and fields by their `@id` for exploration.

**Note:** All entities must be referenced using their `@id` fields per Croissant schema.

Let's enumerate the available record sets and within them, the corresponding fields and column `@id`s.

In [ ]:
print("\nAvailable record sets in this Croissant package:")
for recset in dataset.record_sets:
    print(f"- Record Set @id: {recset.id}, name: {getattr(recset, 'name', None)}")
    for field in recset.fields:
        print(f"  - Field @id: {field.id}, name: {getattr(field, 'name', None)}")
        if hasattr(field, 'column') and field.column:
            # field.column can be a string or an object
            if isinstance(field.column, list):
                for col in field.column:
                    if hasattr(col, 'id'):
                        print(f"    - Column @id: {col.id}, name: {getattr(col, 'name', None)}")
                    else:
                        print(f"    - Column @id: {col}")
            else:
                col = field.column
                if hasattr(col, 'id'):
                    print(f"    - Column @id: {col.id}, name: {getattr(col, 'name', None)}")
                else:
                    print(f"    - Column @id: {col}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

> **You can inspect and change `record_sets_ids` below to include the actual @id values you wish to explore.**

In [ ]:
# First, compile all record sets' @id values:
record_sets_ids = [recset.id for recset in dataset.record_sets]

if not record_sets_ids:
    print("No record sets were found in the Croissant package. Please check the dataset definition.")
else:
    print("Extracting data for record sets:", record_sets_ids)
    dataframes = {}
    for record_set_id in record_sets_ids:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
    # For demonstration, let's pick the first available record set
    target_record_set = record_sets_ids[0]
    # Show columns and a sample
    print("\nFirst few columns for record set @id:", target_record_set)
    print(dataframes[target_record_set].columns.tolist())
    dataframes[target_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply EDA steps: filtering, normalization, and grouping on available fields in a selected record set. All fields must be referenced using their `@id` values.

In [ ]:
import numpy as np

# Check if there is any data to analyze
if not record_sets_ids:
    print("EDA cannot be performed: No record sets present.")
else:
    df = dataframes[target_record_set]
    print(f"Sample size: {len(df)}")
    # Attempt to select numeric fields by checking dtypes or by heuristic
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_fields:
        print("No numeric fields detected -- cannot run numeric EDA.")
    else:
        numeric_field_id = numeric_fields[0]  # using first numeric column for demonstration
        print(f"Using numeric field @id: {numeric_field_id}")

        # Simple filter: greater than median
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print("\nAfter normalization:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping on a non-numeric (categorical-like) field
        non_numeric_fields = [col for col in df.columns if col not in numeric_fields]
        group_field = non_numeric_fields[0] if non_numeric_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nGrouped by {group_field} (mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No non-numeric field for grouping found.")

## 5. Visualization
Let us visualize the distribution of the selected numeric field, and the group averages if a grouping field exists.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_sets_ids or not numeric_fields:
    print("No numeric data to plot.")
else:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, color='cornflowerblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field exists and was used in grouping, plot group means
    if group_field:
        plt.figure(figsize=(10,5))
        # Take top categories if there are too many
        vals = grouped_df.sort_values(ascending=False)
        vals = vals.head(20)
        sns.barplot(x=vals.index.astype(str), y=vals.values, palette="viridis")
        plt.xticks(rotation=70)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
- We loaded dataset metadata, including name, description, and author `@id`s.
- All exploration of record sets, fields, and columns was referenced by their Croissant `@id`.
- We extracted records from available record sets and performed sample EDA: filtering, normalization, and grouping.
- Data visualization provided insight into the distribution of key numeric variables and their relationship with categorical groupings.

**Next steps:**
- Consult domain experts for interpreting key fields and model results
- Further refine EDA and apply modeling techniques on preprocessed data
- Ensure proper handling of sensitive fields listed (`Gender`, `Socio-economic status`, `Age`, `Geography`) in downstream tasks